# Create DSWE Water Masks for RivMapper v2

Generates per-reach, per-year binary water masks from Landsat imagery via
Google Earth Engine, then cleans them locally to retain only
channel-connected pixels.

## What changed from v1

- **GRWL dropped entirely.** Mean width per reach now comes from SWORD nodes
  (~200 m resolution) within each reach polygon. The cleaning buffer is
  built from the v2 reach line (`_reach_lines.gpkg`) buffered by half-width.
- **Pre-compute area, tile preemptively** when a reach exceeds the GEE export
  size limit — no wasted full-size attempts.
- **No `time.sleep(10)` after every export.** geemap exports are synchronous.
- **Skip already-processed (reach, year) pairs** unless `overwrite=True`.
- **Raster-based cleaning** replaces the polygonize-then-spatial-join pipeline
  (orders of magnitude faster on large reaches).
- **Master CSV uses separate range columns**: `year_start`, `year_end`,
  `reach_start`, `reach_end` (no string parsing).

## Workflow per river

1. Load v2 reach polygons, v2 reach lines, and SWORD nodes from disk.
2. For each reach in `[reach_start, reach_end]`:
   - Spatial-join SWORD nodes to the reach polygon → mean width.
   - Build cleaning buffer = reach_line.buffer(mean_width / 2).
   - Pre-compute area; decide if export should be tiled.
   - For each year in `[year_start, year_end]`:
     - Skip if processed output already exists (unless `overwrite=True`).
     - Build percentile-reduced DSWE water mask in EE, threshold to binary.
     - Export to local UTM at 30 m. Tile if pre-computed area > threshold.
     - Stitch tiles if needed; then clean using the rasterized buffer.

## Outputs

```
{working_directory}\RiverMapping\RiverMasks\{river_name}\reach_{ds_order}\
    ├── Raw\        {river_name}_reach_{ds}_{year}_DSWE_level_{lvl}.tif
    └── Processed\  {river_name}_reach_{ds}_{year}_DSWE_level_{lvl}.tif
```

## Required master CSV columns

`river_name`, `working_directory`, `DSWE_threshold_level`, `year_start`,
`year_end`, `reach_start`, `reach_end`, `min_pixels`. Optional:
`mask_percentile` (default 50).

## References

Jones, J.W., 2019. *Improved Automated Detection of Subpixel-Scale Inundation
— Revised Dynamic Surface Water Extent (DSWE) Partial Surface Water Tests.*
Remote Sensing 11, 374.

Greenberg et al., 2024. *A Generalized Area-Based Framework to Quantify River
Mobility From Remotely Sensed Imagery.* JGR Earth Surface 128, e2023JF007189.


In [1]:
import ee
import geemap
import geopandas as gpd
import numpy as np
import pandas as pd
import rasterio
import rasterio.features
import math
import glob
import os
from pathlib import Path
from scipy.ndimage import label as ndi_label
from shapely.geometry import mapping
from tqdm.notebook import tqdm

In [2]:
# ── CONFIG ───────────────────────────────────────────────────────────
CONFIG = {
    "master_csv_path": Path(
        r"E:\Dissertation\Data\Template_RivMapper_master.csv"
    ),
    # Your Earth Engine project ID (e.g., "ee-huckrees").
    "gee_project": "phd-work2023",
    # Max area (km²) per export tile. Reaches larger than this are tiled
    # before the first GEE call.
    "tile_size_km2": 400.0,
    # Default DSWE composite percentile when not specified in the master CSV.
    # 50 = median.
    "default_percentile": 50,
    # Connected-component structure for 8-connected pixel neighborhoods.
    "min_pixels_default": 200,
    # Re-process already-completed (reach, year) pairs.
    "overwrite": False,
}

# ── Constants ────────────────────────────────────────────────────────
# Standardized band names expected by DSWE math.
STANDARD_BANDS = ["uBlue", "Blue", "Green", "Red", "Nir", "Swir1", "Swir2", "BQA"]

# Native Landsat pixel scale (m).
LANDSAT_SCALE_M = 30

# ── Initialize Earth Engine ─────────────────────────────────────────
ee.Initialize(project=CONFIG["gee_project"])
print(f"GEE initialized with project: {CONFIG['gee_project']}")

GEE initialized with project: phd-work2023


In [3]:
def utm_epsg_for_gdf(gdf):
    """
    Return "EPSG:xxxxx" for the UTM zone containing the centroid of
    `gdf`'s bounding box.
    """
    if gdf.crs is None or not gdf.crs.is_geographic:
        gdf_geo = gdf.to_crs("EPSG:4326")
    else:
        gdf_geo = gdf
    minx, miny, maxx, maxy = gdf_geo.total_bounds
    lon = (minx + maxx) / 2.0
    lat = (miny + maxy) / 2.0
    utm_zone = int((lon + 180.0) / 6.0) + 1
    epsg = (32600 if lat >= 0 else 32700) + utm_zone
    return f"EPSG:{epsg}"

In [4]:
# ── DSWE index helpers (operate on ee.Image) ─────────────────────────

def _mndwi(image):
    return image.normalizedDifference(["Green", "Swir1"]).rename("mndwi")


def _mbsrv(image):
    return image.select(["Green"]).add(image.select(["Red"])).rename("mbsrv")


def _mbsrn(image):
    return image.select(["Nir"]).add(image.select(["Swir1"])).rename("mbsrn")


def _ndvi(image):
    return image.normalizedDifference(["Nir", "Red"]).rename("ndvi")


def _awesh(image):
    return image.expression(
        "Blue + 2.5 * Green + (-1.5) * mbsrn + (-0.25) * Swir2",
        {
            "Blue": image.select(["Blue"]),
            "Green": image.select(["Green"]),
            "mbsrn": _mbsrn(image).select(["mbsrn"]),
            "Swir2": image.select(["Swir2"]),
        },
    ).rename("awesh")


def dswe(image):
    """Jones (2019) DSWE classification: 0=none, 1=high, 2=mod, 3=wetland, 4=low."""
    mndwi = _mndwi(image)
    mbsrv = _mbsrv(image)
    mbsrn = _mbsrn(image)
    awesh = _awesh(image)
    swir1 = image.select(["Swir1"])
    nir = image.select(["Nir"])
    ndvi = _ndvi(image)
    blue = image.select(["Blue"])
    swir2 = image.select(["Swir2"])

    t1 = mndwi.gt(0.124)
    t2 = mbsrv.gt(mbsrn)
    t3 = awesh.gt(0)
    t4 = (mndwi.gt(-0.44)
          .And(swir1.lt(0.09)).And(nir.lt(0.15)).And(ndvi.lt(0.7)))
    t5 = (mndwi.gt(-0.5)
          .And(blue.lt(0.1)).And(swir1.lt(0.3))
          .And(swir2.lt(0.1)).And(nir.lt(0.25)))

    t = (t1.add(t2.multiply(10)).add(t3.multiply(100))
         .add(t4.multiply(1000)).add(t5.multiply(10000)))

    no_water = t.eq(0).Or(t.eq(1)).Or(t.eq(10)).Or(t.eq(100)).Or(t.eq(1000))
    h_water = (t.eq(1111).Or(t.eq(10111)).Or(t.eq(11011))
               .Or(t.eq(11101)).Or(t.eq(11110)).Or(t.eq(11111)))
    m_water = (t.eq(111).Or(t.eq(1011)).Or(t.eq(1101)).Or(t.eq(1110))
               .Or(t.eq(10011)).Or(t.eq(10101)).Or(t.eq(10110))
               .Or(t.eq(11001)).Or(t.eq(11010)).Or(t.eq(11100)))
    p_wetland = t.eq(11000)
    l_water = (t.eq(11).Or(t.eq(101)).Or(t.eq(110)).Or(t.eq(1001))
               .Or(t.eq(1010)).Or(t.eq(1100)).Or(t.eq(10000))
               .Or(t.eq(10001)).Or(t.eq(10010)).Or(t.eq(10100)))

    return (no_water.multiply(0)
            .add(h_water.multiply(1))
            .add(m_water.multiply(2))
            .add(p_wetland.multiply(3))
            .add(l_water.multiply(4))
            .rename(["dswe"]))


def classify_water(image, max_level):
    """Binary water mask: True where DSWE class in [1, max_level]."""
    d = dswe(image)
    water = d.eq(1)
    for level in range(2, max_level + 1):
        water = water.Or(d.eq(level))
    return water.rename(["waterMask"])

In [5]:
# ── Landsat collection setup (cached at notebook scope) ──────────────

def _mask_clouds(image):
    """Mask clouds and shadows using Landsat Collection-2 QA_PIXEL bits."""
    cloud_shadow_bit = 1 << 3
    cloud_bit = 1 << 5
    qa = image.select("BQA")
    keep = (qa.bitwiseAnd(cloud_shadow_bit).eq(0)
            .And(qa.bitwiseAnd(cloud_bit).eq(0)))
    return image.updateMask(keep)


def _rescale_sr(image):
    """Collection-2 surface reflectance scale + offset."""
    rescaled = image.select(STANDARD_BANDS).multiply(0.0000275).add(-0.2)
    return image.addBands(rescaled, None, True)


# Band aliasing across Landsat sensors.
_BN89 = ["SR_B1", "SR_B2", "SR_B3", "SR_B4", "SR_B5", "SR_B6", "SR_B7", "QA_PIXEL"]
_BN57 = ["SR_B1", "SR_B1", "SR_B2", "SR_B3", "SR_B4", "SR_B5", "SR_B7", "QA_PIXEL"]

# Build the merged, band-aligned Landsat 5/7/8/9 collection ONCE.
# In v1 this was rebuilt per (reach, year); GEE is lazy but the rebuild
# still triggers metadata churn.
_LS5 = ee.ImageCollection("LANDSAT/LT05/C02/T1_L2").select(_BN57, STANDARD_BANDS)
_LS7 = (ee.ImageCollection("LANDSAT/LE07/C02/T1_L2")
        .filterDate("1999-04-15", "2003-05-30")
        .select(_BN57, STANDARD_BANDS))
_LS8 = ee.ImageCollection("LANDSAT/LC08/C02/T1_L2").select(_BN89, STANDARD_BANDS)
_LS9 = ee.ImageCollection("LANDSAT/LC09/C02/T1_L2").select(_BN89, STANDARD_BANDS)

LANDSAT_COLLECTION = (_LS5.merge(_LS7).merge(_LS8).merge(_LS9)
                      .map(_mask_clouds).map(_rescale_sr))

print("Landsat collection (5/7/8/9, cloud-masked, rescaled) cached at notebook scope.")

Landsat collection (5/7/8/9, cloud-masked, rescaled) cached at notebook scope.


In [6]:
class NoImageryError(RuntimeError):
    """Raised when no Landsat scenes are available for a year × geometry."""


def build_water_mask(year, max_level, ee_geom, percentile=50):
    imagery = (LANDSAT_COLLECTION
               .filterDate(f"{year}-01-01", f"{year}-12-31")
               .filterBounds(ee_geom))
    if imagery.size().getInfo() == 0:
        raise NoImageryError(f"no Landsat scenes for year={year}")

    binary = imagery.map(lambda img: classify_water(img, max_level).rename("water_mask"))
    return (binary.toBands()
            .reduce(ee.Reducer.percentile([percentile]))
            .gte(0.5)                                  # ← NEW: force binary
            .clip(ee_geom)
            .rename("water_mask"))

In [7]:
def shapely_to_ee_geometry(geom, source_crs):
    """
    Convert a shapely Polygon/MultiPolygon (in `source_crs`) to an
    ee.Geometry. Reprojects to EPSG:4326 first since EE geometries are
    expressed in WGS84 by default.

    Avoids v1's pattern of pushing a FeatureCollection to EE and pulling
    it back via getInfo() just to extract individual geometries.
    """
    # Reproject via geopandas (handles the math).
    gdf = gpd.GeoDataFrame(geometry=[geom], crs=source_crs).to_crs("EPSG:4326")
    g = gdf.geometry.iloc[0]
    geojson = mapping(g)
    if geojson["type"] == "Polygon":
        return ee.Geometry.Polygon(geojson["coordinates"])
    if geojson["type"] == "MultiPolygon":
        return ee.Geometry.MultiPolygon(geojson["coordinates"])
    raise ValueError(f"Unsupported geometry type: {geojson['type']}")

In [8]:
def tile_polygon_local(shapely_polygon, source_crs, tile_size_km2):
    """
    Pre-tile a polygon into a grid of sub-polygons each ≤ `tile_size_km2`,
    computed locally (no GEE round-trip). Returns a list of shapely
    polygons in `source_crs` (UTM).
    """
    # Polygon area in m² → km².
    area_km2 = shapely_polygon.area / 1.0e6
    if area_km2 <= tile_size_km2:
        return [shapely_polygon]

    n_tiles = math.ceil(area_km2 / tile_size_km2)
    per_side = math.ceil(math.sqrt(n_tiles))
    minx, miny, maxx, maxy = shapely_polygon.bounds
    dx = (maxx - minx) / per_side
    dy = (maxy - miny) / per_side

    from shapely.geometry import box
    tiles = []
    for i in range(per_side):
        for j in range(per_side):
            tile_box = box(minx + i * dx, miny + j * dy,
                           minx + (i + 1) * dx, miny + (j + 1) * dy)
            clipped = tile_box.intersection(shapely_polygon)
            if not clipped.is_empty and clipped.area > 0:
                tiles.append(clipped)
    return tiles


def export_tile_to_disk(ee_geom, output_path, year, max_level, percentile,
                       output_crs):
    """
    Export one tile's water mask to disk in `output_crs` (UTM) at 30 m.

    Returns "ok", "no_imagery", or "failed".
    """
    try:
        mask = build_water_mask(year, max_level, ee_geom, percentile)
        mask = mask.reproject(crs=output_crs, scale=LANDSAT_SCALE_M)
        geemap.ee_export_image(
            mask,
            filename=str(output_path),
            scale=LANDSAT_SCALE_M,
            region=ee_geom.getInfo()["coordinates"],
            crs=output_crs,
            file_per_band=False,
        )
        return "ok" if Path(output_path).exists() else "failed"
    except NoImageryError:
        return "no_imagery"
    except Exception as e:
        msg = str(e)
        print(f"      Export error: {msg[:200]}")
        return "failed"


def stitch_tiles(tile_paths, output_path):
    """Merge a list of tile rasters into a single raster at `output_path`."""
    from rasterio.merge import merge as rio_merge
    datasets = [rasterio.open(p) for p in tile_paths]
    try:
        mosaic, transform = rio_merge(datasets)
        meta = datasets[0].meta.copy()
        meta.update({
            "driver": "GTiff",
            "height": mosaic.shape[1],
            "width": mosaic.shape[2],
            "transform": transform,
            "compress": "LZW",
        })
        with rasterio.open(output_path, "w", **meta) as dst:
            dst.write(mosaic)
    finally:
        for d in datasets:
            d.close()
    # Remove the per-tile files after merging.
    for p in tile_paths:
        try:
            Path(p).unlink()
        except OSError:
            pass

In [9]:
def mean_width_from_sword_nodes(reach_polygon, sword_nodes_utm):
    """
    Return mean SWORD `width` (m) of nodes that fall within `reach_polygon`.
    Both inputs must be in the same (UTM) CRS.

    Raises ValueError if no nodes fall in the polygon (shouldn't happen for
    v2 reaches derived from SWORD, but guards against pathological cases).
    """
    nodes_in_reach = sword_nodes_utm[sword_nodes_utm.geometry.within(reach_polygon)]
    if nodes_in_reach.empty:
        # Fallback: nodes within a small buffer of the polygon boundary.
        nodes_in_reach = sword_nodes_utm[
            sword_nodes_utm.geometry.intersects(reach_polygon)
        ]
    if nodes_in_reach.empty:
        raise ValueError("No SWORD nodes found within reach polygon.")

    widths = nodes_in_reach["width"].astype(float)
    # SWORD uses negative sentinel values (e.g., -9999) for missing widths.
    widths = widths[widths > 0]
    if widths.empty:
        raise ValueError("No valid (positive) SWORD widths in reach.")
    return float(widths.mean())


def clean_water_mask(raw_path, cleaning_buffer_geom, reach_polygon_geom,
                    min_pixels, output_path):
    """Now also clips to reach_polygon_geom as final step."""
    with rasterio.open(raw_path) as src:
        raw = src.read(1).astype(np.uint8)
        transform = src.transform
        height, width = raw.shape
        profile = src.profile.copy()

    # Cleaning buffer (channel connectivity).
    buffer_raster = rasterio.features.rasterize(
        [(cleaning_buffer_geom, 1)],
        out_shape=(height, width), transform=transform,
        fill=0, dtype=np.uint8,
    )

    # Reach polygon (hard boundary clip).             ← NEW
    polygon_raster = rasterio.features.rasterize(
        [(reach_polygon_geom, 1)],
        out_shape=(height, width), transform=transform,
        fill=0, dtype=np.uint8,
    )

    labeled, _ = ndi_label(raw, structure=np.ones((3, 3), dtype=int))
    touching = np.unique(labeled[buffer_raster == 1])
    touching = touching[touching != 0]
    sizes = np.bincount(labeled.ravel())
    keep_labels = touching[sizes[touching] >= min_pixels]
    cleaned = np.isin(labeled, keep_labels).astype(np.uint8)

    cleaned = cleaned * polygon_raster                # ← NEW: hard clip

    profile.update(dtype=rasterio.uint8, nodata=0, compress="LZW", count=1)
    with rasterio.open(output_path, "w", **profile) as dst:
        dst.write(cleaned, 1)

In [10]:
def process_river(row, config):
    """
    Generate + clean DSWE water masks for every (reach, year) pair in a
    single river's range. Returns a summary dict.
    """
    river_name = row["river_name"]
    working_directory = Path(row["working_directory"])
    max_level = int(row["DSWE_threshold_level"])
    year_start = int(row["year_start"])
    year_end = int(row["year_end"])
    reach_start = int(row["reach_start"])
    reach_end = int(row["reach_end"])
    min_pixels = int(row.get("min_pixels", config["min_pixels_default"]))
    percentile = int(row.get("mask_percentile", config["default_percentile"]))

    # ── Input paths ─────────────────────────────────────────────────
    reaches_dir = working_directory / "RiverMapping" / "Reaches" / river_name
    polys_path = reaches_dir / f"{river_name}_reach_polygons.gpkg"
    lines_path = reaches_dir / f"{river_name}_reach_lines.gpkg"
    nodes_path = (working_directory / "SWORD" / "Extracted_Rivers" / river_name
                  / f"{river_name}_SWORD_nodes.gpkg")

    for p in (polys_path, lines_path, nodes_path):
        if not p.exists():
            return {"river": river_name, "status": f"failed: missing {p.name}"}

    # ── Load and align CRS ──────────────────────────────────────────
    polys = gpd.read_file(polys_path, engine="pyogrio")
    lines = gpd.read_file(lines_path, engine="pyogrio")
    nodes = gpd.read_file(nodes_path, engine="pyogrio")

    # All three to the reach polygons' UTM CRS.
    utm = utm_epsg_for_gdf(polys)
    polys = polys.to_crs(utm)
    lines = lines.to_crs(utm)
    nodes = nodes.to_crs(utm)

    # ── Output root ─────────────────────────────────────────────────
    output_base = working_directory / "RiverMapping" / "RiverMasks" / river_name
    output_base.mkdir(parents=True, exist_ok=True)

    # ── Reach loop ──────────────────────────────────────────────────
    polys = polys[(polys["ds_order"] >= reach_start) &
                  (polys["ds_order"] <= reach_end)]

    counts = {"ok": 0, "skipped": 0, "no_imagery": 0, "failed": 0}

    for _, reach_row in polys.iterrows():
        ds_order = int(reach_row["ds_order"])
        reach_poly = reach_row.geometry
        reach_line_row = lines[lines["ds_order"] == ds_order]
        if reach_line_row.empty:
            print(f"  reach {ds_order}: no matching line, skipping")
            counts["failed"] += year_end - year_start + 1
            continue
        reach_line = reach_line_row.geometry.iloc[0]

        # Mean width + cleaning buffer (in UTM, meters).
        try:
            mean_w = mean_width_from_sword_nodes(reach_poly, nodes)
        except ValueError as e:
            print(f"  reach {ds_order}: {e}")
            counts["failed"] += year_end - year_start + 1
            continue
        cleaning_buffer = reach_line.buffer(mean_w / 2.0)

        # Pre-compute area, decide tiling upfront.
        tiles = tile_polygon_local(reach_poly, utm, config["tile_size_km2"])
        n_tiles = len(tiles)
        ee_tiles = [shapely_to_ee_geometry(t, utm) for t in tiles]

        reach_root = output_base / f"reach_{ds_order}"
        raw_dir = reach_root / "Raw"
        proc_dir = reach_root / "Processed"
        raw_dir.mkdir(parents=True, exist_ok=True)
        proc_dir.mkdir(parents=True, exist_ok=True)

        years = range(year_start, year_end + 1)
        year_iter = tqdm(years, desc=f"reach {ds_order} (w̄={mean_w:.0f}m, "
                                     f"{n_tiles} tile{'s' if n_tiles > 1 else ''})",
                         leave=False)

        for year in year_iter:
            output_name = (f"{river_name}_reach_{ds_order}_{year}"
                           f"_DSWE_level_{max_level}.tif")
            raw_path = raw_dir / output_name
            proc_path = proc_dir / output_name

            if not config["overwrite"] and proc_path.exists():
                counts["skipped"] += 1
                continue

            # Export raw (possibly tiled).
            tile_paths = []
            statuses = []
            for idx, ee_tile in enumerate(ee_tiles):
                tile_out = (raw_path if n_tiles == 1
                            else raw_dir / f"{output_name[:-4]}_tile_{idx}.tif")
                status = export_tile_to_disk(
                    ee_tile, tile_out, year, max_level, percentile, utm,
                )
                statuses.append(status)
                if status == "ok":
                    tile_paths.append(tile_out)

            # No imagery in any tile → skip year.
            if all(s == "no_imagery" for s in statuses):
                counts["no_imagery"] += 1
                continue
            # Hard failure on all tiles.
            if not tile_paths:
                counts["failed"] += 1
                continue

            # Stitch if multiple tiles.
            if n_tiles > 1:
                stitch_tiles([str(p) for p in tile_paths], str(raw_path))

            if not raw_path.exists():
                counts["failed"] += 1
                continue

            # Clean.
            try:
                clean_water_mask(
                    str(raw_path), cleaning_buffer, reach_poly, min_pixels, str(proc_path),
                )
                counts["ok"] += 1
            except Exception as e:
                print(f"  reach {ds_order} year {year}: clean failed: {e}")
                counts["failed"] += 1

    return {"river": river_name, **counts}

In [11]:
# ── Load and validate master CSV ────────────────────────────────────
master = pd.read_csv(CONFIG["master_csv_path"])
master = master.dropna(subset=["river_name"]).reset_index(drop=True)

required_cols = {
    "river_name", "working_directory", "DSWE_threshold_level",
    "year_start", "year_end", "reach_start", "reach_end", "min_pixels",
}
missing = required_cols - set(master.columns)
if missing:
    raise ValueError(f"Master CSV missing required columns: {missing}")

print(f"Loaded {len(master)} rivers from master CSV.\n")

# ── Process each river ──────────────────────────────────────────────
results = []
for _, row in master.iterrows():
    river_name = row["river_name"]
    print(f"\n=== {river_name} ===")
    print(f"  years:   [{row['year_start']}, {row['year_end']}]")
    print(f"  reaches: [{row['reach_start']}, {row['reach_end']}]")
    print(f"  DSWE level ≤ {int(row['DSWE_threshold_level'])}")
    try:
        result = process_river(row, CONFIG)
    except Exception as e:
        result = {"river": river_name, "status": f"failed: {type(e).__name__}: {e}"}
    print(f"  → {result}")
    results.append(result)

# ── Summary ─────────────────────────────────────────────────────────
summary = pd.DataFrame(results).fillna(0)
print("\n──── Summary ────")
print(summary.to_string(index=False))

Loaded 2 rivers from master CSV.


=== OkTedi ===
  years:   [2008.0, 2025.0]
  reaches: [1.0, 4.0]
  DSWE level ≤ 2


reach 1 (w̄=93m, 1 tile):   0%|          | 0/18 [00:00<?, ?it/s]

Generating URL ...
Please wait ...
Data downloaded to E:\Dissertation\Data\RiverMapping\RiverMasks\OkTedi\reach_1\Raw\OkTedi_reach_1_2008_DSWE_level_2.tif
Generating URL ...
Please wait ...
Data downloaded to E:\Dissertation\Data\RiverMapping\RiverMasks\OkTedi\reach_1\Raw\OkTedi_reach_1_2009_DSWE_level_2.tif
Generating URL ...
Please wait ...
Data downloaded to E:\Dissertation\Data\RiverMapping\RiverMasks\OkTedi\reach_1\Raw\OkTedi_reach_1_2010_DSWE_level_2.tif
Generating URL ...
Please wait ...
Data downloaded to E:\Dissertation\Data\RiverMapping\RiverMasks\OkTedi\reach_1\Raw\OkTedi_reach_1_2011_DSWE_level_2.tif
Generating URL ...
Please wait ...
Data downloaded to E:\Dissertation\Data\RiverMapping\RiverMasks\OkTedi\reach_1\Raw\OkTedi_reach_1_2013_DSWE_level_2.tif
Generating URL ...
Please wait ...
Data downloaded to E:\Dissertation\Data\RiverMapping\RiverMasks\OkTedi\reach_1\Raw\OkTedi_reach_1_2014_DSWE_level_2.tif
Generating URL ...
Please wait ...
Data downloaded to E:\Dissertation\

reach 2 (w̄=140m, 1 tile):   0%|          | 0/18 [00:00<?, ?it/s]

Generating URL ...
Please wait ...
Data downloaded to E:\Dissertation\Data\RiverMapping\RiverMasks\OkTedi\reach_2\Raw\OkTedi_reach_2_2008_DSWE_level_2.tif
Generating URL ...
Please wait ...
Data downloaded to E:\Dissertation\Data\RiverMapping\RiverMasks\OkTedi\reach_2\Raw\OkTedi_reach_2_2009_DSWE_level_2.tif
Generating URL ...
Please wait ...
Data downloaded to E:\Dissertation\Data\RiverMapping\RiverMasks\OkTedi\reach_2\Raw\OkTedi_reach_2_2010_DSWE_level_2.tif
Generating URL ...
Please wait ...
Data downloaded to E:\Dissertation\Data\RiverMapping\RiverMasks\OkTedi\reach_2\Raw\OkTedi_reach_2_2011_DSWE_level_2.tif
Generating URL ...
Please wait ...
Data downloaded to E:\Dissertation\Data\RiverMapping\RiverMasks\OkTedi\reach_2\Raw\OkTedi_reach_2_2013_DSWE_level_2.tif
Generating URL ...
Please wait ...
Data downloaded to E:\Dissertation\Data\RiverMapping\RiverMasks\OkTedi\reach_2\Raw\OkTedi_reach_2_2014_DSWE_level_2.tif
Generating URL ...
Please wait ...
Data downloaded to E:\Dissertation\

...

reach 1 (w̄=482m, 4 tiles):   0%|          | 0/18 [00:00<?, ?it/s]

reach 2 (w̄=503m, 4 tiles):   0%|          | 0/18 [00:00<?, ?it/s]

reach 3 (w̄=370m, 4 tiles):   0%|          | 0/18 [00:00<?, ?it/s]

reach 4 (w̄=367m, 4 tiles):   0%|          | 0/18 [00:00<?, ?it/s]

  → {'river': 'Beni', 'ok': 0, 'skipped': 68, 'no_imagery': 4, 'failed': 0}

──── Summary ────
 river  ok  skipped  no_imagery  failed
OkTedi  68        0           4       0
  Beni   0       68           4       0


## Notes

- **Set `gee_project` in CONFIG first.** Replace the placeholder with your
  Earth Engine project ID (`ee-yourname` or whatever shows in
  console.cloud.google.com). The init call will fail loudly if it's still
  the placeholder.

- **First run downloads imagery; subsequent runs reuse cached outputs.**
  The skip-if-processed check looks at the cleaned output path. Set
  `overwrite=True` in CONFIG to force re-processing.

- **`min_pixels` threshold.** Filters speckle by component size. v1's
  default was 200 pixels (≈ 180,000 m² at 30 m). For very small rivers
  consider lowering; for large braided systems raising it prevents oxbow
  fragments from being kept as separate channels.

- **`mask_percentile`.** 50 = median (default). Use a higher percentile
  (e.g., 75) to bias toward wetter conditions, lower to bias toward
  drier. The composite reduces all in-year scenes by this percentile.

- **Tile size.** Default 400 km² is conservative; GEE's actual export
  limit is around 32 MB compressed, which at 30 m and uint8 is roughly
  10× this. Bump `tile_size_km2` to 1000+ for typical reaches.

- **Cleaning buffer width.** `mean_width / 2` on each side gives a total
  belt of one mean channel width. For braided rivers where the active
  channel belt is much wider than mean channel width, this may clip out
  legitimate secondary channels. If that happens, consider buffering by
  `2 × mean_width` or using `max_width` from SWORD nodes instead.

## Next steps

- Test on one short year range (e.g., 2 years) for one reach before
  unleashing the full record.
- If too many years come back as `no_imagery` for early-record reaches,
  consider raising `year_start` or accepting that pre-1999 coverage in
  some regions is just sparse.
- Inspect a Raw + Processed pair side-by-side in QGIS to confirm the
  cleaning buffer is the right shape and the size threshold is reasonable.
